# Phase 1 — Master modeling table

Joins **weather**, **NLCD fractional cover**, and **transmission distance** into one cell-level table for Phase 2+ modeling.

| Input | Output |
|-------|--------|
| `data/processed/weather_cell_features.parquet` | `data/processed/modeling_master_cell.parquet` |
| `data/processed/grid_nlcd_fractional.parquet` | |
| `data/processed/grid_transmission_distance.parquet` | |
| Population files *(optional; unused — filter is off by default)* | |

**Join key:** `(latitude, longitude)`

Adds **`in_ny_state`** via spatial join to NY outline (cached GeoJSON under `data/ref/`).

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import polars as pl
import yaml

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
WEATHER_PATH = ROOT / 'data' / 'processed' / 'weather_cell_features.parquet'
NLCD_PATH = ROOT / 'data' / 'processed' / 'grid_nlcd_fractional.parquet'
TRANSMISSION_PATH = ROOT / 'data' / 'processed' / 'grid_transmission_distance.parquet'
GEOID_PATH = ROOT / 'data' / 'processed' / 'cell_county_geoid.parquet'
POPULATION_PATH = ROOT / 'data' / 'processed' / 'county_population_density.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'modeling_master_cell.parquet'

USE_POPULATION = False  # unused; keep False so results stay identical

JOIN_KEYS = ['latitude', 'longitude']

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

for path, label in [
    (WEATHER_PATH, '01_ingest_weather.ipynb'),
    (NLCD_PATH, '02_nlcd_zonal_stats.ipynb'),
    (TRANSMISSION_PATH, '04_grid_transmission_distance.ipynb'),
]:
    if not path.exists():
        raise FileNotFoundError(f'Run {label} first: {path}')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

weather = pl.read_parquet(WEATHER_PATH).drop_nulls(subset=JOIN_KEYS)
nlcd = pl.read_parquet(NLCD_PATH)
transmission = pl.read_parquet(TRANSMISSION_PATH).select(
  JOIN_KEYS + ['dist_to_transmission_km']
)

print('Weather rows     :', len(weather))
print('NLCD rows        :', len(nlcd))
print('Transmission rows:', len(transmission))

Weather rows     : 4401
NLCD rows        : 4401
Transmission rows: 4401


In [2]:
nlcd_frac_cols = [c for c in nlcd.columns if c.startswith('frac_class_')]
nlcd_keep = JOIN_KEYS + ['nlcd_pixel_count'] + nlcd_frac_cols

master = (
    weather
    .join(nlcd.select(nlcd_keep), on=JOIN_KEYS, how='left', validate='1:1')
    .join(transmission, on=JOIN_KEYS, how='left', validate='1:1')
)

if USE_POPULATION and GEOID_PATH.exists():
    geoid = pl.read_parquet(GEOID_PATH).with_columns(
        pl.col('GEOID').cast(pl.Utf8).str.zfill(5)
    )
    master = master.join(geoid, on=JOIN_KEYS, how='left', validate='1:1')
    print('Joined cell GEOID from 04b')
elif USE_POPULATION:
    print('USE_POPULATION=True but no cell_county_geoid.parquet — run 04b first')
else:
    print('Population join skipped (USE_POPULATION=False)')

if USE_POPULATION and POPULATION_PATH.exists() and 'GEOID' in master.columns:
    population = pl.read_csv(POPULATION_PATH).with_columns(
        pl.col('GEOID').cast(pl.Utf8).str.zfill(5)
    )
    pop_cols = ['GEOID', 'population', 'population_density']
    population = population.select([c for c in pop_cols if c in population.columns])
    master = master.with_columns(pl.col('GEOID').cast(pl.Utf8).str.zfill(5)).join(
        population, on='GEOID', how='left'
    )
    print('Joined county population density')
elif USE_POPULATION and POPULATION_PATH.exists():
    print('Population CSV present but master has no GEOID — skipped')

# NY state geolocation flag (used in Phase 2–3)
import geopandas as gpd

NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'
NY_BOUNDARY_PATH.parent.mkdir(parents=True, exist_ok=True)
NY_STATES_URL = (
    'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'
)

if NY_BOUNDARY_PATH.exists():
    ny_boundary = gpd.read_file(NY_BOUNDARY_PATH)
else:
    states = gpd.read_file(NY_STATES_URL)
    ny_boundary = states[states['name'] == 'New York'].copy()
    ny_boundary.to_file(NY_BOUNDARY_PATH, driver='GeoJSON')

pdf = master.to_pandas()
gdf = gpd.GeoDataFrame(
    pdf,
    geometry=gpd.points_from_xy(pdf['longitude'], pdf['latitude']),
    crs='EPSG:4326',
)
in_ny = gpd.sjoin(gdf, ny_boundary[['geometry']], predicate='within', how='inner')
ny_keys = pl.from_pandas(in_ny[['latitude', 'longitude']].drop_duplicates()).with_columns(
    pl.lit(True).alias('in_ny_state')
)
master = (
    master.join(ny_keys, on=JOIN_KEYS, how='left')
    .with_columns(pl.col('in_ny_state').fill_null(False))
)
print(f'in_ny_state=True: {master["in_ny_state"].sum():,} / {len(master):,}')

master.write_parquet(OUTPUT_PATH)
print(f'Wrote {len(master):,} rows x {len(master.columns)} cols -> {OUTPUT_PATH}')

Population join skipped (USE_POPULATION=False)


in_ny_state=True: 1,725 / 4,401
Wrote 4,401 rows x 103 cols -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/modeling_master_cell.parquet


In [3]:
# QA summary
master.with_columns(
    pl.sum_horizontal([pl.col(c) for c in nlcd_frac_cols]).alias('nlcd_frac_sum')
).select([
    pl.len().alias('n_cells'),
    pl.col('dist_to_transmission_km').null_count().alias('missing_transmission'),
    pl.col('nlcd_pixel_count').null_count().alias('missing_nlcd'),
    (pl.col('nlcd_pixel_count') == 0).sum().alias('nlcd_zero_pixels'),
    pl.col('nlcd_frac_sum').mean().alias('mean_nlcd_frac_sum'),
    pl.col('nlcd_frac_sum').filter(pl.col('nlcd_pixel_count') > 0).mean().alias('mean_frac_sum_where_pixels'),
])

n_cells,missing_transmission,missing_nlcd,nlcd_zero_pixels,mean_nlcd_frac_sum,mean_frac_sum_where_pixels
u32,u32,u32,u32,f64,f64
4401,0,0,703,0.840264,1.0


In [4]:
master.head(3)

latitude,longitude,shortwave_mean,shortwave_mean_24h,shortwave_interannual_std,shortwave_intraseasonal_std_mean,direct_mean,diffuse_mean,shortwave_diurnal_amplitude,clear_sky_index,direct_diffuse_ratio,wind_speed_mean,wind_speed_interannual_std,wind_speed_intraseasonal_std_mean,wind_speed_p90,wind_power_mean,wind_speed_hub_mean,wind_speed_hub_std,wind_hub_power_mean,wind_hub_speed_p50,wind_hub_speed_p75,wind_hub_speed_p90,wind_hub_speed_p95,wind_ramp_p95,weibull_k,weibull_A,temperature_mean,surface_pressure_mean,n_years,year_start,year_end,shortwave_mean_m01,shortwave_mean_m02,shortwave_mean_m03,shortwave_mean_m04,shortwave_mean_m05,shortwave_mean_m06,…,wind_speed_mean_m12,shortwave_mean_DJF,shortwave_mean_MAM,shortwave_mean_JJA,shortwave_mean_SON,wind_speed_mean_DJF,wind_speed_mean_MAM,wind_speed_mean_JJA,wind_speed_mean_SON,wind_power_mean_DJF,wind_power_mean_MAM,wind_power_mean_JJA,wind_power_mean_SON,wind_hub_speed_p50_DJF,wind_hub_speed_p50_JJA,wind_hub_speed_p90_DJF,wind_hub_speed_p90_JJA,wind_data_source,nlcd_pixel_count,frac_class_11,frac_class_12,frac_class_21,frac_class_22,frac_class_23,frac_class_24,frac_class_31,frac_class_41,frac_class_42,frac_class_43,frac_class_52,frac_class_71,frac_class_81,frac_class_82,frac_class_90,frac_class_95,dist_to_transmission_km,in_ny_state
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32,i32,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
41.85,-72.42,308.084773,164.453651,10.726929,241.597051,106.127083,58.326568,968.375,0.468552,1.881404,3.350913,0.147335,1.551727,5.4519808,65.189832,4.483279,2.076099,156.12687,4.174565,5.673169,7.294355,8.379069,1.564502,2.293051,5.060428,10.211249,992.533432,8,2018,2025,194.068145,243.078278,320.376564,334.67874,375.286649,380.407452,…,3.623664,201.39643,345.331964,376.051941,267.780072,3.673426,3.671852,2.791497,3.272425,84.666089,82.931874,34.577023,58.899474,4.625914,3.606523,7.973518,5.832621,"""grid_10km""",170770,0.022732,0.0,0.203865,0.110242,0.042027,0.007191,0.000773,0.4065,0.008093,0.041407,0.001048,0.003882,0.056374,0.019863,0.072021,0.003982,4.06011,false
41.22,-71.97,326.402946,174.254233,9.582264,252.731561,116.108793,58.145441,969.125,0.490363,2.040113,6.645459,0.250052,3.139405,10.863449,512.748771,8.891143,4.200296,1228.011469,8.391084,11.457872,14.534508,16.557503,2.556959,2.240111,10.03825,11.425797,1016.571981,8,2018,2025,201.279435,245.074259,335.817557,355.853728,396.525171,407.974734,…,7.895069,206.860936,364.7481,403.845986,284.382641,7.723579,6.798463,5.187846,6.89554,759.639367,515.771091,225.564624,555.189346,9.978303,6.666913,16.467406,11.003484,"""grid_10km""",170766,0.945229,0.0,0.01823,0.006196,0.001341,0.000299,0.005171,0.012473,0.0,0.001839,0.000053,0.002793,0.000902,0.0,0.002553,0.002922,16.222946,false
41.22,-72.15,322.539828,172.19187,10.229151,249.714065,113.603537,58.588332,968.25,0.485369,1.97887,3.976458,0.15561,1.875507,6.434191,110.909812,5.320213,2.509293,265.624276,4.997109,6.746204,8.608482,9.864757,1.75081,2.244161,6.006545,11.647399,1016.614547,8,2018,2025,201.916532,247.077668,332.30633,350.20982,391.120324,401.983566,…,4.417689,207.640094,359.834154,396.857418,281.691414,4.397776,4.219007,3.308112,3.989323,150.910396,125.240839,58.746103,109.489925,5.546874,4.266237,9.556712,6.96449,"""grid_10km""",170766,0.980342,0.0,0.001792,0.000738,0.000381,0.000146,0.003227,0.008257,0.0,0.000029,0.000041,0.001265,0.000533,0.0,0.001517,0.001733,9.922183,false
